## NBA 3.1 | RX Patient Features

Builds First Occurrence, Recency, and Frequency features for RX claims.

**Input files (via `INPUT_PATHS_31` from `00_event_list_config`):**
- First Occurrence + Recency: `rx_all` (`rx_input_claims_total_events_and_first_occ_features_input`, 4.03B rows, 2021-01-02→2026-08-31)
- Frequency: `rx_12m` (`rx_input_claims_12M_features_input`, 2.55B rows, 2023-05-01→2026-08-31)

**Output:** `/Volumes/.../kerendia_nba_3_1_volume/notebook_output/patient_features/rx/`

**Skill:** `databricks-skills/patient_features_skills/SKILL.md` (Sections 3–5 incl. NBA 3.1 removal notes). Builders come from `00_Patient_Feature_Helpers_31` (optimized, validated). Section 13 add-on features are built in `01_Additional_Patient_Features`.

**NBA 3.1 changes applied:** censored first-occurrence events dropped / capped at 365 d + `_FIRST_CLAIM_BEFORE_WINDOW` flag; unsuffixed recency totals never written; `LOGRATIO_FREQ_*` (N = 3, 6) replaces `CHANGE_IN_FREQ_*`; unique patients for 90/180/360 d only; `FREQ_*_IN_LAST_*` dropped where `NUM_OF_PATIENTS_*` exists (`DROP_FREQ_WITH_PATIENT_EQUIV`). Frequency output = one row per HCP x cohort at `SVC_DT_M = LOOKBACK_END_DATE`.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/03_patient_features/Final Notebooks/00_Patient_Feature_Helpers_31"

In [0]:
import re
from pyspark.sql import functions as F, Window

# ── Feature family (claims only — do not change) ──────────────────────────────
FEATURE_FAMILY = "claims"
CLAIM_TYPE     = "rx"
EVENT_TYPE_VAL = "rx"

# ── Input paths (set by 00_event_list_config) ────────────────────────────────
RX_ALL_PATH  = INPUT_PATHS_31["rx_all"]   # First Occ + Recency
RX_12M_PATH  = INPUT_PATHS_31["rx_12m"]  # Frequency

# ── Output paths ─────────────────────────────────────────────────────────────
OUTPUT_BASE = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
               "kerendia_nba_3/kerendia_nba_3_1_volume/notebook_output/patient_features/rx")
FO_OUT    = f"{OUTPUT_BASE}/rx_first_occurrence_features"
REC_OUT   = f"{OUTPUT_BASE}/rx_recency_features"
FREQ_OUT  = f"{OUTPUT_BASE}/rx_frequency_features"

# ── Test-run sample (driven by job parameter; 0 = full universe) ─────────────
dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0=full)")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))
if TEST_HCP_SAMPLE > 0:   # sample runs write to their own folder so full-universe outputs are never overwritten
    OUTPUT_BASE = f"{OUTPUT_BASE}/_test_sample_{TEST_HCP_SAMPLE}"
    FO_OUT, REC_OUT, FREQ_OUT = [f"{OUTPUT_BASE}/{p.rsplit('/', 1)[1]}" for p in (FO_OUT, REC_OUT, FREQ_OUT)]

# ── Lag / window parameters ──────────────────────────────────────────────────
RECENCY_WINDOWS   = [30, 60, 90, 180, 270, 360]  # days
LAG_WINDOWS       = [1, 2, 3, 6, 12]             # months (Freq Sub-B)
CHANGE_WINDOWS    = [3, 6]                        # months for log-ratio (Sub-C; NBA 3.1)
PATIENT_WINDOWS   = [90, 180, 360]                # days (Freq Sub-D) — NBA 3.1 removal note: 90/180/360 only
DROP_FREQ_WITH_PATIENT_EQUIV = True               # NBA 3.1: drop FREQ_*_IN_LAST_* when a NUM_OF_PATIENTS_* equivalent exists
TMP_BASE          = f"{OUTPUT_BASE}/_tmp"         # Delta checkpoints for intermediates (Section 7.4), removed at the end

print(f"OUTPUT_BASE : {OUTPUT_BASE}")
print(f"RX_ALL_PATH : {RX_ALL_PATH}")
print(f"RX_12M_PATH : {RX_12M_PATH}")

In [0]:
cohorts = CohortGenerator(FEATURE_FAMILY).load()

# Section 10.8.6 checks
meta = cohorts.attrs["metadata"]
assert FEATURE_FAMILY == "claims" and meta["feature_family"] == "claims" and meta["offset_days"] == 0, \
    f"Wrong feature family or offset in cohort metadata: {meta}"
assert list(cohorts["cohort_id"]) == [f"c{i}" for i in range(1, 29)], "Expected 28 cohorts c1..c28"
c1 = cohorts.iloc[0]
assert (str(c1["lookback_start_date"]), str(c1["lookback_end_date"])) == ("2023-06-01", "2024-05-01"), \
    f"c1 dates mismatch: {c1['lookback_start_date']} / {c1['lookback_end_date']}"

print(f"Cohorts loaded: {len(cohorts)} rows")
print(cohorts[["cohort_id", "lookback_start_date", "lookback_end_date", "dataset"]].to_string(index=False))

In [0]:
HCP_UNIVERSE_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                     "kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe")
hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()

if TEST_HCP_SAMPLE > 0:
    hcp_universe = (hcp_universe
                    .orderBy(F.xxhash64("PROVIDER_ID"), "PROVIDER_ID")
                    .limit(TEST_HCP_SAMPLE))

hcp_universe = F.broadcast(hcp_universe)
rx_event_keys = event_groups("rx")
print(f"Universe HCPs : {hcp_universe.count() if TEST_HCP_SAMPLE > 0 else '314,920 (full)'}")
print(f"Rx EVENT_GROUPs: {len(rx_event_keys)}")

In [0]:
claims_all = (spark.read.format("delta").load(RX_ALL_PATH)
              .join(hcp_universe, "PROVIDER_ID", "left_semi"))
claims_all = standardize(claims_all, EVENT_TYPE_VAL)
validate_event_input(claims_all, CLAIM_TYPE)

# Early prune — keep only columns needed for First Occ + Recency
claims_all = claims_all.select(
    "PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE", "EVENT_TYPE", "CLAIM_ID")

In [0]:
# Pre-aggregate -> pivot -> cross-join cohorts, plus the NBA 3.1 censoring rule (Section 3 removal note):
#   >50% of HCPs first seen within 60 d of the data start -> dropped; <=10% -> raw; else min(days, 365) + flag
fo, fo_censor_report = build_first_occurrence(claims_all, rx_event_keys, cohorts, tmp_path=f"{TMP_BASE}/_fo_pre")

print(fo_censor_report["RULE"].value_counts().to_string())
print("Dropped (left-censored):", fo_censor_report.loc[fo_censor_report["RULE"] == "DROP_CENSORED", "EVENT_SAN"].tolist())
(spark.createDataFrame(fo_censor_report.astype({"DATA_START": str}))
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .save(f"{OUTPUT_BASE}/rx_first_occurrence_censor_report"))

In [0]:
(fo.write.format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .partitionBy("COHORT")
   .save(FO_OUT))
print(f"Written: {FO_OUT}")

In [0]:
# One daily pre-aggregate + conditional aggregation (7 cohorts per pass) instead of 28 cohorts x 14 shuffles.
# Validated against the old loop: identical values for all 237 recency columns on a 300-HCP sample.
recency = build_recency(claims_all, rx_event_keys, cohorts, suffix="RX", windows=RECENCY_WINDOWS,
                        count_claim_ids=True, tmp_base=f"{TMP_BASE}/rec")

In [0]:
(recency.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("COHORT")
        .save(REC_OUT))
print(f"Written: {REC_OUT}")

In [0]:
claims_12m = (spark.read.format("delta").load(RX_12M_PATH)
               .join(hcp_universe, "PROVIDER_ID", "left_semi"))
claims_12m = standardize(claims_12m, EVENT_TYPE_VAL)
claims_12m = (claims_12m
              .withColumn("SVC_DT_M", F.trunc(F.col("EVENT_DATE"), "month"))
              .select("PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "SVC_DT_M", "CLAIM_ID"))
print(f"claims_12m rows (sample): {claims_12m.limit(5).count()} (lazy)")

In [0]:
# Optimized Frequency (Section 5):
#   Sub-A  monthly counts kept in LONG format (no 258-column pivot per month)
#   Sub-B  rolling 3/6/12-month sums computed only at each cohort snapshot month, all 28 cohorts in ONE
#          conditional aggregation (no dense HCP x month spine, no 1,290-column window)
#   Sub-C  LOGRATIO_FREQ_* for N = 3, 6
#   Sub-D  unique patients 90/180/360 d via a per-patient month bitmask: one pass instead of 28 x 5 scans
#   Small dimensions (event map, 28-row cohort table, HCP universe) are broadcast.
# Output grain: one row per HCP x cohort at SVC_DT_M = LOOKBACK_END_DATE (the skeleton join row, Section 8.2).
frequency = build_frequency(claims_12m, rx_event_keys, cohorts, patient_prefix="PRESCRIBED_WITH",
                            lag_windows=LAG_WINDOWS, change_windows=CHANGE_WINDOWS,
                            patient_windows=PATIENT_WINDOWS,
                            drop_freq_with_patient_equiv=DROP_FREQ_WITH_PATIENT_EQUIV,
                            tmp_base=f"{TMP_BASE}/freq")
(frequency.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .partitionBy("COHORT").save(FREQ_OUT))
print(f"Written: {FREQ_OUT} | columns: {len(frequency.columns)}")

In [0]:
for path, label in [(FO_OUT, "First Occurrence"), (REC_OUT, "Recency"), (FREQ_OUT, "Frequency")]:
    df = spark.read.format("delta").load(path)
    cohort_ids = {r[0] for r in df.select("COHORT").distinct().collect()}
    expected   = set(cohorts["cohort_id"])
    extra      = cohort_ids - expected
    missing    = expected - cohort_ids
    n_outside  = (df.select("PROVIDER_ID").distinct()
                    .join(hcp_universe, "PROVIDER_ID", "left_anti").count())
    print(f"{label}: {df.count():,} rows | cohorts={len(cohort_ids)}/28 "
          f"| extra={extra or 'none'} | missing={missing or 'none'} "
          f"| HCPs outside universe={n_outside}")
dbutils.fs.rm(TMP_BASE, True)   # drop intermediate checkpoints
dbutils.notebook.exit("RX patient features written: First Occurrence, Recency, Frequency")